In [ ]:
!pip install -q spacy pandas

!python -m spacy download en_core_web_sm

print("Stage 2 libraries installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Stage 2 libraries installed.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:


from pathlib import Path
import json
import re
from collections import Counter

QUALITY_JSON = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage1_results/"
    "20260728_044011_"
    "honesty_when_a_child_finds_a_lost_wallet_"
    "quality_control.json"
)

if not QUALITY_JSON.exists():
    raise FileNotFoundError(
        f"Quality-control file was not found: {QUALITY_JSON}"
    )

with QUALITY_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    stage1_record = json.load(file)

final_status = stage1_record[
    "human_review"
]["final_status"]

if final_status != "APPROVED":
    raise ValueError(
        "The Stage 1.1 story is not approved. "
        f"Current status: {final_status}"
    )

TOPIC = stage1_record["topic"]
AGE_GROUP = stage1_record["age_group"]

final_story_for_stage2 = stage1_record[
    "stage2_input"
]

print("Stage 1.1 record loaded successfully.")
print("Final status:", final_status)
print("Topic:", TOPIC)
print("Age group:", AGE_GROUP)
print(
    "Story characters:",
    len(final_story_for_stage2),
)

print("\nSTAGE 2 INPUT STORY")
print("=" * 80)
print(final_story_for_stage2)

Stage 1.1 record loaded successfully.
Final status: APPROVED
Topic: honesty when a child finds a lost wallet
Age group: 8–10
Story characters: 1458

STAGE 2 INPUT STORY
It was a sunny day in the park when Emily noticed something shiny on the ground. She picked it up and discovered a small leather wallet. Inside were some money, a business card and an identification card. Emily knew that taking something that belonged to another person would be wrong. She decided to ask her friend Timmy to help her find the owner of the lost wallet.

Emily and Timmy examined the business card and found a telephone number. With help from Emily's mother, they called the number. A worried man named Mr. Johnson answered. He explained that he had lost his wallet while walking through the park earlier that day. Emily's mother arranged for everyone to meet at the nearby community centre, where the wallet could be returned safely.

When Mr. Johnson arrived with his wife, Mrs. Johnson, he correctly described the

In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully.")

spaCy model loaded successfully.


In [ ]:
BAD_PRONOUNS = {
    "he", "she", "they", "him", "her",
    "them", "it", "this", "that",
    "these", "those",
}

STOP_ANSWERS = {
    "a sunny day",
    "one sunny day",
    "one day",
    "today",
    "tomorrow",
    "yesterday",
    "in the morning",
    "in the afternoon",
    "in the evening",
    "at the end of the day",
    "the end of the day",
}


def normalize_text(text: str) -> str:
    """Normalize text for comparison and duplicate detection."""

    text = (text or "").lower().strip()
    text = re.sub(r"[^a-z0-9 ?!]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def good_answer(answer: str) -> bool:
    """Reject answer spans that are unsuitable for QG."""

    normalized_answer = normalize_text(answer)

    if not normalized_answer:
        return False

    if normalized_answer in BAD_PRONOUNS:
        return False

    tokens = normalized_answer.split()

    if len(tokens) == 0 or len(tokens) > 8:
        return False

    if len(normalized_answer) < 3:
        return False

    if normalized_answer in STOP_ANSWERS:
        return False

    return True


def span_salience_score(
    story: str,
    span: str,
    entity_label: str = "",
) -> float:
    """
    Score an answer span using position, entity type and length.

    A larger score means the span is considered more useful.
    """

    normalized_story = normalize_text(story)
    normalized_span = normalize_text(span)

    position = normalized_story.find(normalized_span)

    if position < 0:
        position_bonus = 0.0
    else:
        position_bonus = max(
            0.0,
            1.0 - position / max(1, len(normalized_story)),
        )

    entity_bonus = 0.0

    if entity_label == "PERSON":
        entity_bonus = 0.25
    elif entity_label in {"GPE", "LOC", "FAC"}:
        entity_bonus = 0.20
    elif entity_label in {"ORG", "PRODUCT"}:
        entity_bonus = 0.15

    token_count = len(normalized_span.split())

    if 2 <= token_count <= 4:
        length_bonus = 0.15
    elif token_count == 1:
        length_bonus = 0.05
    else:
        length_bonus = 0.0

    return position_bonus + entity_bonus + length_bonus


def candidate_answers_ranked(
    story: str,
    max_candidates: int = 200,
) -> list[tuple]:
    """Extract and rank named entities and noun phrases."""

    document = nlp(story)
    candidates = []

    # Named entities: people, places, dates and organisations.
    for entity in document.ents:
        answer = entity.text.strip()

        if answer in story and good_answer(answer):
            candidates.append(
                (answer, entity.label_)
            )

    # Noun phrases: wallet, business card, telephone number, etc.
    for noun_chunk in document.noun_chunks:
        answer = noun_chunk.text.strip()

        if answer in story and good_answer(answer):
            candidates.append(
                (answer, "")
            )

    # Remove normalized duplicates.
    seen = set()
    scored_candidates = []

    for answer, entity_label in candidates:
        normalized_answer = normalize_text(answer)

        if normalized_answer in seen:
            continue

        seen.add(normalized_answer)

        score = span_salience_score(
            story,
            answer,
            entity_label,
        )

        scored_candidates.append(
            (answer, entity_label, score)
        )

    scored_candidates.sort(
        key=lambda item: item[2],
        reverse=True,
    )

    return scored_candidates[:max_candidates]


print("Answer-span ranking functions are ready.")

Answer-span ranking functions are ready.


In [ ]:
ranked_spans = candidate_answers_ranked(
    final_story_for_stage2,
    max_candidates=100,
)

span_rows = []

for rank, (
    answer,
    entity_label,
    score,
) in enumerate(ranked_spans, start=1):

    span_rows.append(
        {
            "rank": rank,
            "answer_span": answer,
            "entity_label": (
                entity_label
                if entity_label
                else "NOUN_PHRASE"
            ),
            "salience_score": round(score, 4),
        }
    )

ranked_spans_df = pd.DataFrame(span_rows)

print(
    "Total unique candidate spans:",
    len(ranked_spans_df),
)

display(ranked_spans_df.head(20))

Total unique candidate spans: 45


,rank,answer_span,entity_label,salience_score
0,1,Emily,PERSON,1.2748
1,2,the park,NOUN_PHRASE,1.1346
2,3,the ground,NOUN_PHRASE,1.1016
3,4,Timmy,PERSON,1.0819
4,5,a small leather wallet,NOUN_PHRASE,1.0715
5,6,some money,NOUN_PHRASE,1.0469
6,7,a business card,NOUN_PHRASE,1.0392
7,8,an identification card,NOUN_PHRASE,1.0252
8,9,something,NOUN_PHRASE,1.0149
9,10,another person,NOUN_PHRASE,0.9740


In [ ]:
GENERIC_ANSWERS = {
    "something",
    "someone",
    "somebody",
    "anything",
    "anyone",
    "everything",
    "everyone",
    "one",
    "another person",
    "a person",
    "the person",
    "people",
    "help",
    "stuff",
    "things",
    "it",
}


def canonical_answer(answer: str) -> str:
    """
    Normalize an answer for stronger duplicate detection.

    For example:
    'a business card' and 'the business card'
    both become 'business card'.
    """

    normalized = normalize_text(answer)

    tokens = normalized.split()

    if tokens and tokens[0] in {
        "a", "an", "the",
    }:
        tokens = tokens[1:]

    return " ".join(tokens)


def is_strong_answer_candidate(answer: str) -> bool:
    """Apply stronger filtering to an extracted answer span."""

    normalized = normalize_text(answer)

    if not good_answer(answer):
        return False

    if normalized in GENERIC_ANSWERS:
        return False

    tokens = normalized.split()

    if not tokens:
        return False

    # Reject phrases beginning with unclear pronouns.
    if tokens[0] in {
        "he", "she", "they", "his",
        "her", "their", "this", "that",
    }:
        return False

    return True


def select_diverse_answer_spans(
    ranked_candidates: list[tuple],
    maximum_spans: int = 12,
) -> list[tuple]:
    """
    Select strong and reasonably diverse answer spans.

    Returns:
    answer, entity label and salience score.
    """

    selected = []
    seen_canonical_answers = set()

    for answer, entity_label, score in ranked_candidates:

        if not is_strong_answer_candidate(answer):
            continue

        canonical = canonical_answer(answer)

        if canonical in seen_canonical_answers:
            continue

        seen_canonical_answers.add(canonical)

        selected.append(
            (answer, entity_label, score)
        )

        if len(selected) >= maximum_spans:
            break

    return selected


print("Strong and diverse span selector is ready.")

Strong and diverse span selector is ready.


In [ ]:
selected_spans = select_diverse_answer_spans(
    ranked_spans,
    maximum_spans=12,
)

selected_span_rows = []

for rank, (
    answer,
    entity_label,
    score,
) in enumerate(selected_spans, start=1):

    selected_span_rows.append(
        {
            "rank": rank,
            "selected_answer": answer,
            "type": (
                entity_label
                if entity_label
                else "NOUN_PHRASE"
            ),
            "score": round(score, 4),
        }
    )

selected_spans_df = pd.DataFrame(
    selected_span_rows
)

print(
    "Selected answer spans:",
    len(selected_spans_df),
)

display(selected_spans_df)

Selected answer spans: 12


,rank,selected_answer,type,score
0,1,Emily,PERSON,1.2748
1,2,the park,NOUN_PHRASE,1.1346
2,3,the ground,NOUN_PHRASE,1.1016
3,4,Timmy,PERSON,1.0819
4,5,a small leather wallet,NOUN_PHRASE,1.0715
5,6,some money,NOUN_PHRASE,1.0469
6,7,a business card,NOUN_PHRASE,1.0392
7,8,an identification card,NOUN_PHRASE,1.0252
8,9,Johnson,PERSON,0.9417
9,10,the owner,NOUN_PHRASE,0.9158


In [ ]:
def find_support_sentence(
    story: str,
    answer: str,
) -> tuple[int, str]:
    """Find the sentence containing an answer span."""

    sentences = [
        sentence.text.strip()
        for sentence in nlp(story).sents
        if sentence.text.strip()
    ]

    normalized_answer = normalize_text(answer)

    for index, sentence in enumerate(sentences):
        if normalized_answer in normalize_text(sentence):
            return index, sentence

    return -1, ""


def expand_person_title(
    story: str,
    answer: str,
    entity_label: str,
) -> str:
    """Expand Johnson to Mr. Johnson when it appears that way."""

    if entity_label != "PERSON":
        return answer

    title_pattern = re.compile(
        rf"\b(?:Mr|Mrs|Ms|Dr)\.\s+{re.escape(answer)}\b",
        flags=re.IGNORECASE,
    )

    match = title_pattern.search(story)

    if match:
        return match.group(0)

    return answer


def answer_concept_key(
    answer: str,
    entity_label: str,
) -> str:
    """
    Create a concept key to remove answers referring
    to the same main object.
    """

    document = nlp(answer)

    if entity_label == "PERSON":
        return canonical_answer(answer)

    useful_tokens = [
        token.lemma_.lower()
        for token in document
        if (
            not token.is_stop
            and not token.is_punct
            and token.pos_ in {
                "NOUN",
                "PROPN",
            }
        )
    ]

    if useful_tokens:
        # Use the final main noun:
        # leather wallet and lost wallet both become wallet.
        return useful_tokens[-1]

    return canonical_answer(answer)


def select_coverage_aware_spans(
    story: str,
    ranked_candidates: list[tuple],
    maximum_spans: int = 12,
    maximum_per_sentence: int = 2,
) -> list[dict]:
    """
    Select strong spans while reducing repetition and
    improving coverage across the story.
    """

    selected = []
    seen_concepts = set()
    sentence_counts = Counter()

    for answer, entity_label, score in ranked_candidates:

        if not is_strong_answer_candidate(answer):
            continue

        expanded_answer = expand_person_title(
            story,
            answer,
            entity_label,
        )

        sentence_index, support_sentence = (
            find_support_sentence(
                story,
                expanded_answer,
            )
        )

        if sentence_index < 0:
            continue

        if (
            sentence_counts[sentence_index]
            >= maximum_per_sentence
        ):
            continue

        concept_key = answer_concept_key(
            expanded_answer,
            entity_label,
        )

        if concept_key in seen_concepts:
            continue

        seen_concepts.add(concept_key)
        sentence_counts[sentence_index] += 1

        selected.append(
            {
                "answer": expanded_answer,
                "entity_label": (
                    entity_label
                    if entity_label
                    else "NOUN_PHRASE"
                ),
                "score": round(score, 4),
                "support_sentence_index": sentence_index,
                "support_sentence": support_sentence,
                "concept_key": concept_key,
            }
        )

        if len(selected) >= maximum_spans:
            break

    return selected


print("Coverage-aware span selector is ready.")

Coverage-aware span selector is ready.


In [ ]:
final_selected_spans = select_coverage_aware_spans(
    story=final_story_for_stage2,
    ranked_candidates=ranked_spans,
    maximum_spans=12,
    maximum_per_sentence=2,
)

final_selected_spans_df = pd.DataFrame(
    final_selected_spans
)

final_selected_spans_df.insert(
    0,
    "rank",
    range(1, len(final_selected_spans_df) + 1),
)

print(
    "Final selected spans:",
    len(final_selected_spans_df),
)

display(
    final_selected_spans_df[
        [
            "rank",
            "answer",
            "entity_label",
            "score",
            "support_sentence_index",
            "concept_key",
        ]
    ]
)

Final selected spans: 12


,rank,answer,entity_label,score,support_sentence_index,concept_key
0,1,Emily,PERSON,1.2748,0,emily
1,2,the park,NOUN_PHRASE,1.1346,0,park
2,3,Timmy,PERSON,1.0819,4,timmy
3,4,a small leather wallet,NOUN_PHRASE,1.0715,1,wallet
4,5,some money,NOUN_PHRASE,1.0469,2,money
5,6,a business card,NOUN_PHRASE,1.0392,2,card
6,7,Mr. Johnson,PERSON,0.9417,7,mr johnson
7,8,the owner,NOUN_PHRASE,0.9158,4,owner
8,9,a telephone number,NOUN_PHRASE,0.8583,5,number
9,10,Emily's mother,NOUN_PHRASE,0.8344,6,mother


In [ ]:
GENERIC_ROLE_ANSWERS = {
    "the owner",
    "an owner",
    "a worried man",
    "the worried man",
    "a man",
    "the man",
    "a woman",
    "the woman",
}


def select_balanced_story_spans(
    story: str,
    ranked_candidates: list[tuple],
    maximum_spans: int = 12,
) -> list[dict]:
    """
    Select answer spans from the beginning, middle
    and ending of the story.
    """

    sentences = [
        sentence.text.strip()
        for sentence in nlp(story).sents
        if sentence.text.strip()
    ]

    total_sentences = len(sentences)

    section_names = {
        0: "beginning",
        1: "middle",
        2: "ending",
    }

    # For 12 answers, aim for four from each section.
    section_limit = max(
        1,
        maximum_spans // 3,
    )

    section_counts = Counter()
    seen_concepts = set()
    selected = []

    for answer, entity_label, score in ranked_candidates:

        if not is_strong_answer_candidate(answer):
            continue

        if normalize_text(answer) in GENERIC_ROLE_ANSWERS:
            continue

        expanded_answer = expand_person_title(
            story,
            answer,
            entity_label,
        )

        sentence_index, support_sentence = (
            find_support_sentence(
                story,
                expanded_answer,
            )
        )

        if sentence_index < 0:
            continue

        # Divide sentence positions into three sections.
        section_number = min(
            2,
            (sentence_index * 3)
            // max(1, total_sentences),
        )

        section_name = section_names[
            section_number
        ]

        if (
            section_counts[section_name]
            >= section_limit
        ):
            continue

        concept_key = answer_concept_key(
            expanded_answer,
            entity_label,
        )

        if concept_key in seen_concepts:
            continue

        seen_concepts.add(concept_key)
        section_counts[section_name] += 1

        selected.append(
            {
                "answer": expanded_answer,
                "entity_label": (
                    entity_label
                    if entity_label
                    else "NOUN_PHRASE"
                ),
                "score": round(score, 4),
                "story_section": section_name,
                "support_sentence_index": sentence_index,
                "support_sentence": support_sentence,
                "concept_key": concept_key,
            }
        )

    # Sort the final selection by story section and score.
    section_order = {
        "beginning": 0,
        "middle": 1,
        "ending": 2,
    }

    selected.sort(
        key=lambda item: (
            section_order[item["story_section"]],
            -item["score"],
        )
    )

    return selected[:maximum_spans]


print("Balanced story-span selector is ready.")

Balanced story-span selector is ready.


In [ ]:
balanced_selected_spans = (
    select_balanced_story_spans(
        story=final_story_for_stage2,
        ranked_candidates=ranked_spans,
        maximum_spans=12,
    )
)

balanced_spans_df = pd.DataFrame(
    balanced_selected_spans
)

balanced_spans_df.insert(
    0,
    "rank",
    range(1, len(balanced_spans_df) + 1),
)

print(
    "Balanced selected spans:",
    len(balanced_spans_df),
)

print(
    "\nSection distribution:"
)

print(
    balanced_spans_df[
        "story_section"
    ].value_counts()
)

display(
    balanced_spans_df[
        [
            "rank",
            "answer",
            "entity_label",
            "story_section",
            "support_sentence_index",
            "score",
        ]
    ]
)

Balanced selected spans: 12

Section distribution:
story_section
beginning    4
middle       4
ending       4
Name: count, dtype: int64


,rank,answer,entity_label,story_section,support_sentence_index,score
0,1,Emily,PERSON,beginning,0,1.2748
1,2,the park,NOUN_PHRASE,beginning,0,1.1346
2,3,the ground,NOUN_PHRASE,beginning,0,1.1016
3,4,Timmy,PERSON,beginning,4,1.0819
4,5,Mr. Johnson,PERSON,middle,7,0.9417
5,6,Emily's mother,NOUN_PHRASE,middle,6,0.8344
6,7,the number,NOUN_PHRASE,middle,6,0.8155
7,8,Mr. Johnson,NOUN_PHRASE,middle,7,0.7938
8,9,the community,NOUN_PHRASE,ending,15,0.3239
9,10,an honest and responsible decision,NOUN_PHRASE,ending,13,0.2931


In [ ]:
WEAK_WH_ANSWERS = {
    "what",
    "who",
    "where",
    "when",
    "why",
    "how",
    "which",
}


def select_balanced_story_spans_v2(
    story: str,
    ranked_candidates: list[tuple],
    maximum_spans: int = 12,
    maximum_per_sentence: int = 2,
) -> list[dict]:
    """
    Select diverse answer spans from the beginning,
    middle and ending while removing duplicate text.
    """

    sentences = [
        sentence.text.strip()
        for sentence in nlp(story).sents
        if sentence.text.strip()
    ]

    total_sentences = len(sentences)

    section_names = {
        0: "beginning",
        1: "middle",
        2: "ending",
    }

    section_limit = max(
        1,
        maximum_spans // 3,
    )

    section_counts = Counter()
    sentence_counts = Counter()

    seen_concepts = set()
    seen_answers = set()

    selected = []

    for answer, entity_label, score in ranked_candidates:

        normalized_answer = normalize_text(answer)

        if not is_strong_answer_candidate(answer):
            continue

        if normalized_answer in GENERIC_ROLE_ANSWERS:
            continue

        if normalized_answer in WEAK_WH_ANSWERS:
            continue

        expanded_answer = expand_person_title(
            story,
            answer,
            entity_label,
        )

        canonical = canonical_answer(
            expanded_answer
        )

        # Prevent Mr. Johnson appearing twice under
        # different spaCy labels.
        if canonical in seen_answers:
            continue

        sentence_index, support_sentence = (
            find_support_sentence(
                story,
                expanded_answer,
            )
        )

        if sentence_index < 0:
            continue

        # Prevent too many answers from one sentence.
        if (
            sentence_counts[sentence_index]
            >= maximum_per_sentence
        ):
            continue

        section_number = min(
            2,
            (sentence_index * 3)
            // max(1, total_sentences),
        )

        section_name = section_names[
            section_number
        ]

        if (
            section_counts[section_name]
            >= section_limit
        ):
            continue

        concept_key = answer_concept_key(
            expanded_answer,
            entity_label,
        )

        if concept_key in seen_concepts:
            continue

        seen_answers.add(canonical)
        seen_concepts.add(concept_key)

        sentence_counts[sentence_index] += 1
        section_counts[section_name] += 1

        selected.append(
            {
                "answer": expanded_answer,
                "entity_label": (
                    entity_label
                    if entity_label
                    else "NOUN_PHRASE"
                ),
                "score": round(score, 4),
                "story_section": section_name,
                "support_sentence_index": sentence_index,
                "support_sentence": support_sentence,
                "concept_key": concept_key,
            }
        )

    section_order = {
        "beginning": 0,
        "middle": 1,
        "ending": 2,
    }

    selected.sort(
        key=lambda item: (
            section_order[item["story_section"]],
            -item["score"],
        )
    )

    return selected[:maximum_spans]


print("Refined balanced selector is ready.")

Refined balanced selector is ready.


In [ ]:
final_stage2_spans = (
    select_balanced_story_spans_v2(
        story=final_story_for_stage2,
        ranked_candidates=ranked_spans,
        maximum_spans=12,
        maximum_per_sentence=2,
    )
)

final_stage2_spans_df = pd.DataFrame(
    final_stage2_spans
)

final_stage2_spans_df.insert(
    0,
    "rank",
    range(1, len(final_stage2_spans_df) + 1),
)

print(
    "Final Stage 2 spans:",
    len(final_stage2_spans_df),
)

print("\nSection distribution:")

print(
    final_stage2_spans_df[
        "story_section"
    ].value_counts()
)

print(
    "\nDuplicate answers:",
    final_stage2_spans_df[
        "answer"
    ].str.lower().duplicated().sum(),
)

display(
    final_stage2_spans_df[
        [
            "rank",
            "answer",
            "entity_label",
            "story_section",
            "support_sentence_index",
            "score",
        ]
    ]
)

Final Stage 2 spans: 12

Section distribution:
story_section
beginning    4
middle       4
ending       4
Name: count, dtype: int64

Duplicate answers: 0


,rank,answer,entity_label,story_section,support_sentence_index,score
0,1,Emily,PERSON,beginning,0,1.2748
1,2,the park,NOUN_PHRASE,beginning,0,1.1346
2,3,Timmy,PERSON,beginning,4,1.0819
3,4,a small leather wallet,NOUN_PHRASE,beginning,1,1.0715
4,5,Mr. Johnson,PERSON,middle,7,0.9417
5,6,Emily's mother,NOUN_PHRASE,middle,6,0.8344
6,7,the number,NOUN_PHRASE,middle,6,0.8155
7,8,earlier that day,DATE,middle,8,0.7292
8,9,the community,NOUN_PHRASE,ending,15,0.3239
9,10,an honest and responsible decision,NOUN_PHRASE,ending,13,0.2931


In [ ]:
TIME_WORDS = {
    "day", "night", "morning", "evening",
    "afternoon", "today", "tomorrow",
    "yesterday", "week", "month", "year",
}

LOCATION_WORDS = {
    "park", "school", "home", "house",
    "library", "centre", "center",
    "hospital", "shop", "store",
    "street", "garden", "community",
}


def infer_expected_question_type(
    answer: str,
    entity_label: str,
) -> str:
    """Infer a suitable WH-question type from an answer."""

    normalized_answer = normalize_text(answer)
    answer_words = set(normalized_answer.split())

    if entity_label == "PERSON":
        return "who"

    if entity_label in {"GPE", "LOC", "FAC"}:
        return "where"

    if entity_label in {"DATE", "TIME"}:
        return "when"

    if answer_words.intersection(TIME_WORDS):
        return "when"

    if answer_words.intersection(LOCATION_WORDS):
        return "where"

    return "what"


for span_record in final_stage2_spans:
    span_record["expected_question_type"] = (
        infer_expected_question_type(
            span_record["answer"],
            span_record["entity_label"],
        )
    )


stage2_output_df = pd.DataFrame(
    final_stage2_spans
)

stage2_output_df.insert(
    0,
    "rank",
    range(1, len(stage2_output_df) + 1),
)

display(
    stage2_output_df[
        [
            "rank",
            "answer",
            "expected_question_type",
            "story_section",
            "support_sentence_index",
        ]
    ]
)

,rank,answer,expected_question_type,story_section,support_sentence_index
0,1,Emily,who,beginning,0
1,2,the park,where,beginning,0
2,3,Timmy,who,beginning,4
3,4,a small leather wallet,what,beginning,1
4,5,Mr. Johnson,who,middle,7
5,6,Emily's mother,what,middle,6
6,7,the number,what,middle,6
7,8,earlier that day,when,middle,8
8,9,the community,where,ending,15
9,10,an honest and responsible decision,what,ending,13


In [ ]:
from datetime import datetime, timezone

STAGE2_RESULTS_DIR = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage2_results"
)

STAGE2_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

safe_topic = re.sub(
    r"[^a-z0-9]+",
    "_",
    TOPIC.lower(),
).strip("_")[:50]

stage2_result_path = STAGE2_RESULTS_DIR / (
    f"{timestamp}_{safe_topic}_answer_spans.json"
)

stage2_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_stage1_file": str(QUALITY_JSON),
    "topic": TOPIC,
    "age_group": AGE_GROUP,
    "story": final_story_for_stage2,
    "stage1_status": final_status,
    "selection_method": {
        "extractor": "spaCy named entities and noun chunks",
        "ranking": (
            "position, entity-type and length salience"
        ),
        "coverage": (
            "balanced beginning-middle-ending selection"
        ),
        "maximum_spans": 12,
        "maximum_per_sentence": 2,
    },
    "candidate_count_before_filtering": len(
        ranked_spans
    ),
    "selected_span_count": len(
        final_stage2_spans
    ),
    "selected_spans": final_stage2_spans,
    "stage2_status": "COMPLETE",
}

with stage2_result_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stage2_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Stage 2 result saved:")
print(stage2_result_path)

print("\nStage 2 status: COMPLETE")
print(
    "Selected spans:",
    len(final_stage2_spans),
)

Stage 2 result saved:
/content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage2_results/20260728_102306_honesty_when_a_child_finds_a_lost_wallet_answer_spans.json

Stage 2 status: COMPLETE
Selected spans: 12
